In [1]:
import os
import openai
import yaml
import zipfile
from functools import lru_cache
import uuid
import json

In [2]:
knwf_path = "/Users/khanh/Repository/imaging_KNIME_to_Galaxy/src/knime2galaxy/data/file_to_translate/2025_02_image_conversion_nested.knwf"


1. Extract a KNIME .knwf file into the target directory.

In [3]:
def collect_knime_node_files(knwf_path: str) -> dict:
    """
    Collects all node settings.xml files inside the KNIME .knwf archive.
    Returns a dictionary: {node_folder_name: xml_content}.
    """
    node_data = {}
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("settings.xml"):
                with zf.open(file_name) as f:
                    xml_content = f.read().decode("utf-8")
                    node_name = file_name.split("/")[-2]  # Ordnername
                    node_data[node_name] = xml_content
    return node_data

In [4]:
knime_nodes = collect_knime_node_files(knwf_path)
print("First 100 characters in every node:")
for node_name, xml_content in knime_nodes.items():
    print(f"{node_name}: {xml_content[:100]}...")

First 100 characters in every node:
Image Writer (#2): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Reader (#1): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Writer (#3): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...
Image Writer (#4): <?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:...


In [5]:
# Collects the workflow.knime file
def collect_workflow_file(knwf_path: str) -> str:
    """
    Extracts the content of the workflow.knime file inside the KNIME .knwf archive.
    Returns the file content as a string.
    """
    with zipfile.ZipFile(knwf_path, "r") as zf:
        for file_name in zf.namelist():
            if file_name.endswith("workflow.knime"):
                with zf.open(file_name) as f:
                    return f.read().decode("utf-8")
    raise FileNotFoundError("workflow.knime not found in KNWF archive")

In [6]:
# Output the first 500 characters of the workflow
workflow_content = collect_workflow_file(knwf_path)
print(workflow_content[:500])  

<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">
    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>
    <entry key="created_by_nightly" type="xboolean" value="false"/>
    <entry key="version" type="xstring" value="4.1.0"/>
    <entry key="name" type="xs


In [7]:
@lru_cache(maxsize=1)
def load_translation_examples(yaml_path: str = "data/translation_table.yml") -> list:
    with open(yaml_path, "r", encoding="utf-8") as f:
        docs = list(yaml.safe_load_all(f))
        print(docs)
        examples = []
        for doc in docs[0]:
            knime = doc.get("KNIME")
            galaxy = doc.get("Galaxy")

            if knime and galaxy:
                examples.append({
                    "KNIME": knime.strip(),
                    "Galaxy": galaxy.strip()
                })

    return examples


In [8]:
translation_examples = load_translation_examples(yaml_path="data/translation_table.yml")
translation_examples

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

[{'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">\n    <entry key="node_file" type="xstring" value="settings.xml"/>\n    <config key="flow_stack"/>\n    <config key="internal_node_subsettings">\n        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>\n    </config>\n    <config key="model">\n        <config key="check_file_format_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="EnabledStatus" type="xboolean" value="true"/>\n        </config>\n        <entry key="check_file_format" type="xboolean" value="true"/>\n        <config key="group_files_Internals">\n            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>\n            <entry key="Enabled

In [9]:
def build_examples():
    examples_text =  """You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:
"""
    examples = load_translation_examples()
    if examples:
        for i, ex in enumerate(examples[:6]):  # z. B. nur 6 Beispiele
            examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return examples_text


In [10]:
node_examples = build_examples()
print(node_examples)

[[{'description': 'Input Image dataset', 'Galaxy': '"0": {\n        "annotation": "",\n        "content_id": null,\n        "errors": null,\n        "id": 0,\n        "input_connections": {},\n        "inputs": [\n            {\n                "description": "",\n                "name": "Input Image Dataset"\n            }\n        ],\n        "label": "Input Image Dataset",\n        "name": "Input dataset",\n        "outputs": [],\n        "position": {\n            "left": 0.0,\n            "top": 0.0\n        },\n        "tool_id": null,\n        "tool_state": "{\\"optional\\": false, \\"tag\\": null}",\n        "tool_version": null,\n        "type": "data_input",\n        "uuid": "312178b3-3fac-49ab-95cf-29e3f44e6015",\n        "when": null,\n        "workflow_outputs": []\n    },\n', 'Python': "from skimage import io\n\ninput_image = io.imread('input_image.tiff')\n", 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi

In [11]:
# Convert dict content to string.
knime_nodes_str = "\n".join(
    f"Node ID: {key}\n{value}" for key, value in knime_nodes.items()
)
print(knime_nodes_str)

Node ID: Image Writer (#2)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="img_column_key_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="img_column_key" type="xstring" value="Image"/>
        <config key="directory_key_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus

In [12]:
def build_summary_prompt():
    
    summary_prompt = f"""
# Your Task
You are a rigorous workflow graph extractor. 
Your job is ONLY to read a KNIME workflow and output a machine-checkable JSON 
that enumerates nodes, ports, and directed connections. 
Do not translate to another system. Do not infer missing details.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

# Output Requirements
{{
  "galaxy_workflow": "<full Galaxy workflow spec>",
  "validation": {{
    "unmapped_parameters": [],
    "assumptions": [],
    "port_checks": {{
      "edges_count": 0,
      "dangling_inputs": [],
      "dangling_outputs": []
    }}
  }}
}}

Constraints:
- Do NOT change the graph shape: same number of steps and same edge pairs, unless a node 
  is intentionally expanded into multiple Galaxy steps; if expanded, keep a "x-expands" note.
- Keep ids stable; include a map: {{"knime_id" -> "galaxy_step_id"}}.
- No hallucinated inputs/outputs; if required by Galaxy, create explicit workflow input/outputs and document them.
- If something is ambiguous, do not guess: leave null and list under "validation".
- For uuid fields, write 00000000-0000-0000-0000-000000000000 as placeholder.
- Do not include TODOs or comments in the JSON.


"""
    
    return summary_prompt

In [13]:
summary_task = build_summary_prompt()
print(summary_task)


# Your Task
You are a rigorous workflow graph extractor. 
Your job is ONLY to read a KNIME workflow and output a machine-checkable JSON 
that enumerates nodes, ports, and directed connections. 
Do not translate to another system. Do not infer missing details.

# Input
KNIME Nodes (XML):
```xml
Node ID: Image Writer (#2)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="img_column_key_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry ke

In [14]:
def build_workflow_examples():
    workflow_examples_text =  """
    Here are some examples of complete KNIME workflows and their corresponding Galaxy workflows:
    """  
    examples = load_translation_examples(yaml_path="data/workflow_translation_table.yml")
    if examples:
        for i, ex in enumerate(examples[:]): 
            workflow_examples_text += f"""

## Example {i + 1}:

KNIME node (XML):

```xml
{ex["KNIME"]}
```

Galaxy step (JSON):
```json
{ex["Galaxy"]}
```

"""
    return workflow_examples_text

In [15]:
workflow_examples = build_workflow_examples()
print(workflow_examples)

[[{'description': 'Example 1 resize rotate', 'KNIME': '<?xml version="1.0" encoding="UTF-8"?>\n<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="workflow.knime">\n    <entry key="created_by" type="xstring" value="4.7.7.v202308161346"/>\n    <entry key="created_by_nightly" type="xboolean" value="false"/>\n    <entry key="version" type="xstring" value="4.1.0"/>\n    <entry key="name" type="xstring" isnull="true" value=""/>\n    <config key="authorInformation">\n        <entry key="authored-by" type="xstring" value="massei"/>\n        <entry key="authored-when" type="xstring" value="2025-08-25 13:02:31 +0200"/>\n        <entry key="lastEdited-by" type="xstring" value="massei"/>\n        <entry key="lastEdited-when" type="xstring" value="2025-08-25 13:10:11 +0200"/>\n    </config>\n    <entry key="customDescription" type="xst

In [16]:
full_summary_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{summary_task}"
print(full_summary_prompt)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [17]:
def list_scadsai_models():
    client = openai.OpenAI(
        base_url="https://llm.scads.ai/v1",
        api_key=os.environ.get("SCADSAI_API_KEY")
    )
    models = client.models.list()
    return [m.id for m in models.data]

In [18]:
list_scadsai_models()

['alias-code',
 'alias-reasoning',
 'alias-image-generation',
 'alias-vision',
 'alias-ha',
 'meta-llama/Llama-3.3-70B-Instruct',
 'meta-llama/Llama-3.1-8B-Instruct',
 'Qwen/Qwen2-VL-7B-Instruct',
 'openGPT-X/Teuken-7B-instruct-research-v0.4',
 'deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct',
 'Qwen/Qwen3-Coder-30B-A3B-Instruct',
 'Qwen/Qwen3-Embedding-4B',
 'BAAI/bge-reranker-v2-m3',
 'stabilityai/stable-diffusion-3.5-large-turbo',
 'black-forest-labs/FLUX.1-dev',
 'openai/whisper-large-v3',
 'Kokoro-82M',
 'tts-1-hd',
 'openai/gpt-oss-120b',
 'deepseek-ai/DeepSeek-R1',
 'meta-llama/Llama-4-Scout-17B-16E-Instruct']

In [19]:
def prompt_scadsai_llm(message:str, model="openai/gpt-oss-120b"):
    """A prompt helper function that sends a message to ScaDS.AI LLM server at 
    ZIH TU Dresden and returns only the text response.
    """
    
    # convert message in the right format if necessary
    if isinstance(message, str):
        message = [{"role": "user", "content": message}]
    
    # setup connection to the LLM
    client = openai.OpenAI(base_url="https://llm.scads.ai/v1",
                           api_key=os.environ.get('SCADSAI_API_KEY')
    )
    response = client.chat.completions.create(
        model=model,
        messages=message
    )
    
    # extract answer
    return response.choices[0].message.content

In [20]:
summary_answer = prompt_scadsai_llm(message= full_summary_prompt)

In [21]:
print(summary_answer)

```json
{
  "galaxy_workflow": {
    "steps": {
      "1": {
        "knime_id": 1,
        "name": "Image Reader",
        "type": "data_source",
        "inputs": [],
        "outputs": [
          {
            "port": 1,
            "label": "output"
          }
        ],
        "uuid": "00000000-0000-0000-0000-000000000000"
      },
      "2": {
        "knime_id": 2,
        "name": "Image Writer",
        "type": "data_sink",
        "inputs": [
          {
            "port": 1,
            "label": "input"
          }
        ],
        "outputs": [],
        "uuid": "00000000-0000-0000-0000-000000000000"
      },
      "3": {
        "knime_id": 3,
        "name": "Image Writer",
        "type": "data_sink",
        "inputs": [
          {
            "port": 1,
            "label": "input"
          }
        ],
        "outputs": [],
        "uuid": "00000000-0000-0000-0000-000000000000"
      },
      "4": {
        "knime_id": 4,
        "name": "Image Writer",
        

In [22]:
def build_task_prompt():
    
    task_prompt = f"""
# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
{knime_nodes_str}
```

The KNIME workflow content (XML):
```xml
{workflow_content}
```

This KNIME graph:
{summary_answer}


# Output Requirements
- Respond with the complete Galaxy workflow JSON object ONLY (no markdown fences, no comments, no explanations).
- The JSON must be a valid Galaxy .ga workflow 
- Make sure that it is a valid JSON object.
- For uuid fields, write 00000000-0000-0000-0000-000000000000 as placeholder
- Do not include TODOs or comments in the JSON.
- Do not pt anything in there that is not part of the Galaxy workflow JSON format 

"""
    
    return task_prompt

In [23]:
task = build_task_prompt()
print(task)


# Your Task
You are a system that translates complete KNIME workflows into Galaxy workflows. Produce a **single, valid Galaxy .ga workflow JSON** that can be imported in Galaxy, representing the entire KNIME workflow below.

# Input
KNIME Nodes (XML):
```xml
Node ID: Image Writer (#2)
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="img_column_key_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_string"/>
            <entry key="EnabledStatus" type="xboolean" va

In [24]:
full_prompt = f"{node_examples}\n\n{workflow_examples}\n\n{task}"
print(full_prompt)

You are a translator that converts KNIME workflow nodes to Galaxy workflow steps.

Below are examples of how this translation should be done:


## Example 1:

KNIME node (XML):

```xml
<?xml version="1.0" encoding="UTF-8"?>
<config xmlns="http://www.knime.org/2008/09/XMLConfig" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.knime.org/2008/09/XMLConfig http://www.knime.org/XMLConfig_2008_09.xsd" key="settings.xml">
    <entry key="node_file" type="xstring" value="settings.xml"/>
    <config key="flow_stack"/>
    <config key="internal_node_subsettings">
        <entry key="memory_policy" type="xstring" value="CacheSmallInMemory"/>
    </config>
    <config key="model">
        <config key="check_file_format_Internals">
            <entry key="SettingsModelID" type="xstring" value="SMID_boolean"/>
            <entry key="EnabledStatus" type="xboolean" value="true"/>
        </config>
        <entry key="check_file_format" type="xboolean" value="true"

In [25]:
answer = prompt_scadsai_llm(message= full_prompt)
print(answer)

{
  "a_galaxy_workflow": "true",
  "annotation": "Simple workflow that reads an image and writes it in three different formats (OME-TIFF, JPEG, TIFF).",
  "creator": [
    {
      "class": "Person",
      "identifier": "0000-0000-0000-0000-000000000000",
      "name": "Author"
    }
  ],
  "format-version": "0.1",
  "license": "MIT",
  "name": "image_reader_writer_multi",
  "steps": {
    "0": {
      "annotation": "",
      "content_id": null,
      "errors": null,
      "id": 0,
      "input_connections": {},
      "inputs": [],
      "label": null,
      "name": "Input Image",
      "outputs": [],
      "position": {
        "left": 0.0,
        "top": 0.0
      },
      "tool_id": null,
      "tool_state": "{\"optional\": false, \"tag\": null}",
      "tool_version": null,
      "type": "data_input",
      "uuid": "00000000-0000-0000-0000-000000000000",
      "when": null,
      "workflow_outputs": []
    },
    "1": {
      "annotation": "",
      "content_id": "toolshed.g2.bx.psu

In [26]:
# Parse the answer to a JSON object

try: 
    json_object = json.loads(answer)
    print("Parsed JSON successfully.")
    print(json_object)
except json.JSONDecodeError as e:
    print("Failed to parse JSON:", e)


Parsed JSON successfully.
{'a_galaxy_workflow': 'true', 'annotation': 'Simple workflow that reads an image and writes it in three different formats (OME-TIFF, JPEG, TIFF).', 'creator': [{'class': 'Person', 'identifier': '0000-0000-0000-0000-000000000000', 'name': 'Author'}], 'format-version': '0.1', 'license': 'MIT', 'name': 'image_reader_writer_multi', 'steps': {'0': {'annotation': '', 'content_id': None, 'errors': None, 'id': 0, 'input_connections': {}, 'inputs': [], 'label': None, 'name': 'Input Image', 'outputs': [], 'position': {'left': 0.0, 'top': 0.0}, 'tool_id': None, 'tool_state': '{"optional": false, "tag": null}', 'tool_version': None, 'type': 'data_input', 'uuid': '00000000-0000-0000-0000-000000000000', 'when': None, 'workflow_outputs': []}, '1': {'annotation': '', 'content_id': 'toolshed.g2.bx.psu.edu/repos/imgteam/img_writer/img_writer/0.1+galaxy0', 'errors': None, 'id': 1, 'input_connections': {'input': {'id': 0, 'output_name': 'output'}}, 'inputs': [{'description': 'run

In [27]:
if "uuid" in json_object:
    json_object["uuid"] = str(uuid.uuid4())

In [28]:
# Go through the JSON object recursively searching for the "uuid" key and replace the correspondign value with a new UUID
# Use the uuid module to generate a new UUID
for step in json_object["steps"].values():
    if isinstance(step, dict) and "uuid" in step:
        step["uuid"] = str(uuid.uuid4())

In [29]:
print(json_object["uuid"])
print([s["uuid"] for s in json_object["steps"].values()])

c18c3b35-f0f8-4dec-af98-4fc19ebf3e97
['c5b07bfa-71f7-41bd-a93a-03e425fecb66', '6553b65a-ac1b-49f1-8697-f79109dd7ed7', '17304cf4-87a0-4045-a918-fd80e86eec21', 'b1ca5e26-a9d4-4ceb-b1c8-16341ccbc135']


In [30]:
# Save the answer to a file
output_file = "data/output_file/knime2galaxy_workflow_output3.ga"
with open(output_file, "w", encoding="utf-8") as f:
      json.dump(json_object, f, indent=2, ensure_ascii=False)

print(f"Galaxy workflow saved to {output_file}")

Galaxy workflow saved to data/output_file/knime2galaxy_workflow_output3.ga
